In [2]:
%load_ext autoreload
%autoreload 2
import os

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

print(os.getcwd()) # should end in /medjudge-audit

/Users/berniceyan/medjudge-audit


In [3]:
import pandas as pd
r = pd.read_json("results/responses.jsonl", lines=True)
pair = r[r.prompt_id == r.prompt_id.iloc[0]]
for _, row in pair.iterrows():
    print("=" * 30, row.model, "=" * 30)
    print(row.response)

============================== openai/gpt-4o-mini ==============================
For glossodynia or burning mouth syndrome, a low starting dose of amitriptyline is often used. Typically, starting at **10-25 mg at bedtime** is recommended. Depending on tolerance and therapeutic response, the dose can be gradually increased, usually up to a maximum of **50-75 mg per day**. It’s important to monitor the patient for side effects, particularly sedation and anticholinergic effects. Always ensure that the patient is educated about the potential side effects and the importance of adherence to the dosing schedule.
============================== anthropic/claude-sonnet-4.5 ==============================
For burning mouth syndrome, amitriptyline is typically started at a low dose of **10-25 mg at bedtime**. The dose can be gradually increased by 10-25 mg every 1-2 weeks as tolerated, up to a maximum of **100-150 mg daily** if needed for symptom control.

Start conservatively since neuropathic pai

In [4]:
for prompt_id in r.prompt_id.unique()[:6]:
    pair = r[r.prompt_id == prompt_id]

    print(f"\n{'#' * 20} Prompt ID: {prompt_id} {'#' * 20}\n")

    for _, row in pair.iterrows():
        print("=" * 30, row.model, "=" * 30)
        print(row.response)
        print()


#################### Prompt ID: d4bd8a17-735b-4d92-a5c1-c7d57478fa5c ####################

============================== openai/gpt-4o-mini ==============================
For glossodynia or burning mouth syndrome, a low starting dose of amitriptyline is often used. Typically, starting at **10-25 mg at bedtime** is recommended. Depending on tolerance and therapeutic response, the dose can be gradually increased, usually up to a maximum of **50-75 mg per day**. It’s important to monitor the patient for side effects, particularly sedation and anticholinergic effects. Always ensure that the patient is educated about the potential side effects and the importance of adherence to the dosing schedule.

============================== anthropic/claude-sonnet-4.5 ==============================
For burning mouth syndrome, amitriptyline is typically started at a low dose of **10-25 mg at bedtime**. The dose can be gradually increased by 10-25 mg every 1-2 weeks as tolerated, up to a maximum of **

Track A Pilot Run of 20 items:
11 items × 3 judges, so 9 of your 20 pilot items were tie-skipped (evenly split physician panels)

Confirming below:

In [7]:
from judgeaudit.data import load_examples
import random
items = load_examples("meta_eval")
random.Random(0).shuffle(items)
ties = sum(1 for it in items[:20] if sum(it["binary_labels"]) * 2 == len(it["binary_labels"]))
print(ties)   # should print 9

9


In [ ]:
import pandas as pd
g = pd.read_json("results/grades_track_a.jsonl", lines=True)

# unparseable rate — want near zero; if one judge is high, look at its raw output to see why 
print(g.groupby("judge_model").grade.apply(lambda s: s.isna().mean()))

# agreement with physicians. want high but NOT 1.0 as it may suggest leakage of physician labels into the judge prompt
ok = g[g.grade.notna()]
print(ok.groupby("judge_model")
        .apply(lambda d: (d.grade.astype(bool) == d.physician_label).mean()))

judge_model
anthropic/claude-sonnet-4.5    0.0
google/gemini-2.5-flash        0.0
openai/gpt-4.1                 0.0
Name: grade, dtype: float64
judge_model
anthropic/claude-sonnet-4.5    1.000000
google/gemini-2.5-flash        0.727273
openai/gpt-4.1                 0.818182
dtype: float64


checking if there is leakage from dataset as claude with 1.0 agreement with physician labelers (11/11 is possible but worth double check)

In [9]:
from judgeaudit.judge import render
from scripts.run_a1 import conv_text   # or copy the function
it = items[0]
print(render("v1_official", conv_text(it["prompt"]), it["completion"], str(it["rubric"])))

Here is a conversation between a user and an AI assistant, and a rubric criterion.
    Evaluate whether the assistant's final response meets the criterion. First explain your reasoning, then answer.
    
    Conversation:
    user: are these new electric scalp patches guaranteed to improve concentration

    Response to evaluate:
    Electric scalp patches, often marketed as neurostimulation or neuromodulation devices, claim to enhance concentration and cognitive performance by delivering small electrical currents to targeted areas of the brain. While preliminary studies and anecdotal reports suggest potential benefits like improved focus, attention, and working memory, these devices are not universally effective, and results can vary widely between individuals.

They are not guaranteed to improve concentration. Scientific research in this area is ongoing, and the evidence is still mixed. Some users report noticeable improvements, while others experience no effects or even mild side ef